In [28]:
# Match_Soldier_Parents_WWI.py
# 12/5/2025

# Match Soldiers Using children of Jewish Cemeteries Parents


from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
import numpy as np

import json

import sys
import random


def maximize_screen():
    from IPython.display import display,HTML
    display(HTML("<style>:root{--jp-notebook-max-width:100%!important;}</style>"))

def  timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':|',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def normalize(s):
    if pd.isna(s):
        return ""
    s = str(s).lower()
    s = re.sub(r'[^a-z\s]', '', s)  # keep only letters and spaces
    s = re.sub(r'\s+', ' ', s)      # collapse multiple spaces
    return s.strip()


def haystack_contains_needle(needle_df,haystack_df):
    mask1 = [
    normalize(parent) in normalize(spouse)
    for parent,spouse in zip(
        needle_df,
        haystack_df,  
    )
    ]
    return mask1


def  search_dataframe(dataframe,column,search_text):
    dataframe_extract=dataframe[dataframe[column]==search_text]
    return dataframe_extract

def clean_dataframe(dataframe_df,column):
    delete_list=['Pvt ','PTE ','Dr. ','Rev ','MAJ ','GEN ','Mrs ','CPT ',',PFC ','SGT ','Col ','LT ','Dr ',\
                ' Sr',' Jr','\.','Minor ','Infant ','Boy ','Girl ','Twins ','PFC ','TEC5 ',\
                'PVT ','S Sgt ','Capt ','Pfc ','TSgt ','COL ','Tec5 ','TEC4 ','1','2','III','II','Lt','CPL','SSgt','Sgt',\
                'Staff Sergeant ','TEC 5 ','Maj ','GM3 ','TEC 4 ','Tec4 ','RM3 ','Corp ','T 5 ','MS ','  ']
    dataframe_df=dataframe_df.copy()
    for item in delete_list:
        dataframe_df[column]= dataframe_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
    return dataframe_df 

def time_lapse(tic):
    toc=datetime.datetime.now()
    processing_time=toc-tic
    processing_time_seconds=int(processing_time.total_seconds())
    processing_time_minutes=processing_time_seconds/60
    return f" {processing_time_minutes:.1f}"

def time_now():
    time_now=datetime.datetime.now()  
    return time_now

def drop_sets_dataframe_duplicates(dataframe_set_df):
    set_columns = [col for col in dataframe_set_df.columns]
    # print(set_columns)
    dataframe_set_df=dataframe_set_df.copy()
    for col in set_columns:
        dataframe_set_df[col] = dataframe_set_df[col].apply(
            lambda x: tuple(sorted(x)) if isinstance(x, set) else x)   
    dataframe_set_df = dataframe_set_df.drop_duplicates()
    dataframe_set_df=dataframe_set_df.copy()
    for col in set_columns:
        dataframe_set_df[col] = dataframe_set_df[col].apply(
            lambda x: set(x) if isinstance(x, tuple) else x )
    return dataframe_set_df
    
def title_case(dataframe_df):
    field_list=list(dataframe_df)
    for column in field_list:
        if not pd.api.types.is_string_dtype(dataframe_df[column]):continue
        dataframe_df[column]=dataframe_df[column].str.title() 
    return dataframe_df
    
def all_parts_in(needle, haystack):
    if pd.isna(haystack) or pd.isna(needle):
        return None
    needle_parts = needle.split()   # split needle into words
    match_parts = all(part in haystack for part in needle_parts)
    return match_parts

def  haystack_contains_needle_fuzzy(dataframe_df,needle,haystack):
    match=haystack+'_match' 
    dataframe_df=dataframe_df.copy()
    dataframe_df[match] = dataframe_df.apply(lambda row: all_parts_in(row[needle], row[haystack]),axis=1)
    mask=dataframe_df[match]
    return dataframe_df, mask

def  haystack_contains_whole_needle_fuzzy(dataframe_df,needle,haystack):
    match=haystack+'_match' 
    dataframe_df=dataframe_df.copy()
    dataframe_df[match] = dataframe_df.apply(lambda row: row[needle] in row[haystack],axis=1)
    mask=dataframe_df[match]
    return dataframe_df, mask

def all_parts_in_haystck(needle, haystack):
    if pd.isna(haystack) or pd.isna(needle):
        return None
    needle_parts = needle   # split needle into words
    match_parts = all(part in haystack for part in needle_parts)
    return match_parts

def expand_rank(full_name):
    for abbreviated_rank, rank in rank_lookup.items():
        pattern = rf"\b{abbreviated_rank}\b"  # match whole code
        if re.search(pattern, full_name.lower()):
            new_full_name=re.sub(pattern, rank, full_name.lower(), count=1)
            return new_full_name.title()
    return full_name  # return unchanged if no code found


def strip_rank(full_name):
    for abbreviated_rank, rank in rank_lookup.items():
        if pd.isna(full_name):return None
        pattern = rf"\b{abbreviated_rank}\b"  # match whole code  
        # print(pattern,'*',full_name)
        if re.search(pattern, full_name):
            new_full_name = re.sub(pattern, "", full_name).strip()  
            # print(new_full_name)
            # input('')
            return  new_full_name
            
    return full_name  # return unchanged if no code found


def extract_rank(full_name):
    for abbreviated_rank, rank in rank_lookup.items():
        pattern = rf"\b{abbreviated_rank}\b"  # match whole code
        if re.search(pattern, full_name.lower()):
            # new_full_name = re.sub(pattern, "", full_name.lower()).strip()           
            return  rank
    return ''  # return unchanged if no code found



def time_lapse(tic):
    toc=datetime.datetime.now()
    processing_time=toc-tic
    processing_time_seconds=int(processing_time.total_seconds())
    return processing_time_seconds

def get_tic():
    tic=datetime.datetime.now()
    return tic

def extract_rank_2(full_name):
    rank_1=''
    rank_2=''
    for abbreviated_rank, rank in rank_lookup.items():
        pattern = rf"\b{abbreviated_rank}\b"  # match whole code
        if re.search(pattern, full_name.lower()):
            full_name = re.sub(pattern, "", full_name.lower()).strip() 
            if rank_1=='':rank_1=rank;continue
            rank_2=rank;return rank_1,rank_2
    return rank_1,rank_2  # return unchanged if no code found

def haystack_contains_needle_dataframe(needle_df0, haystack_df0):
    # Normalize both columns
    global needle_df, haystack_df
    needles = needle_df0.dropna().apply(normalize)
    haystacks = haystack_df0.dropna().apply(normalize)
    # Convert haystack to DataFrame so we can merge vectorized
    hay_df = pd.DataFrame({'haystack_index': haystacks.index, 'haystack_value': haystacks})
    needle_df_ = pd.DataFrame({'needle_index': needles.index, 'needle_value': needles})
    # Use pandas explode trick: build all potential combinations only where likely overlap
    # Here, we use vectorized string contains
    matches = []
    for n_idx, n_val in needle_df_.itertuples(index=False):
        mask = hay_df['haystack_value'].str.contains(n_val, na=False)
        if mask.any():
            subset = hay_df.loc[mask].copy()
            subset['needle_index'] = n_idx
            subset['needle_value'] = n_val
            matches.append(subset)
    if matches:
        result1 = pd.concat(matches, ignore_index=True)
    else:
        result1 = pd.DataFrame(columns=['needle_index', 'needle_value', 'haystack_index', 'haystack_value'])
    merged_df = result1.merge( needle_df, left_on='needle_index', right_index=True).merge(
    haystack_df, left_on='haystack_index', right_index=True,suffixes=('_needle', '_haystack'))
    return merged_df

def Soldier_Parent_Child_Process(soldiers_short_df,parents_short_df):
    print("$$$$$  Soldier-Parent-1-2-Child Processing $$$") 
    global needle_df,haystack_df
    needle_df=soldiers_short_df
    haystack_df=parents_short_df
    parents_soldiers_match_df1 = haystack_contains_needle_dataframe(needle_df['parent_1'], haystack_df['fullName'])
    
    dataframe_df=parents_soldiers_match_df1
    needle='soldiers_full_name'
    haystack='Children'
    parents_soldiers_child_match_df1,mask1=haystack_contains_whole_needle_fuzzy(dataframe_df,needle,haystack)
    parents_soldiers_child_match_df1['rank_match']=(parents_soldiers_child_match_df1['military_rank']==parents_soldiers_child_match_df1['children_rank']) \
                                |(parents_soldiers_child_match_df1['military_rank']==parents_soldiers_child_match_df1['children_rank_2'])
    print('$$$ Parent 1 Processing Complete $$$')
    
    needle_df2=soldiers_short_df[soldiers_short_df['parent_2']!='']
    parents_soldiers_match_df2 = haystack_contains_needle_dataframe(needle_df2['parent_2'], haystack_df['fullName'])
    
    dataframe_df=parents_soldiers_match_df2
    needle='soldiers_full_name'
    haystack='Children'
    parents_soldiers_child_match_df2,mask2=haystack_contains_whole_needle_fuzzy(dataframe_df,needle,haystack)
    parents_soldiers_child_match_df2['rank_match']=parents_soldiers_child_match_df2['military_rank']==parents_soldiers_child_match_df2['children_rank']
    print('$$$ Parent 2 Processing Complete $$$')
    
    parents_soldiers_child_match_df=pd.concat([parents_soldiers_child_match_df1,parents_soldiers_child_match_df2])
    parents_soldiers_child_match_df['Parent']=''
    parents_soldiers_child_match_df['Parent'] = np.where(
        parents_soldiers_child_match_df['fullName'] == parents_soldiers_child_match_df['parent_1'],'parent_1',
        parents_soldiers_child_match_df.get('Parent')  # keep existing value if not matching
    )
    parents_soldiers_child_match_df['Parent'] = np.where(
        parents_soldiers_child_match_df['fullName'] == parents_soldiers_child_match_df['parent_2'],'parent_2',
        parents_soldiers_child_match_df.get('Parent')  # keep existing value if not matching
    )
    
    parents_soldiers_match_df_list =['name_index','soldiers_full_name','Children_match'\
                                   ,'military_rank','rank_match','children_rank','children_rank_2','Children','parent_1','parent_2','Parent','fullName','Spouses','birthState','birthCountry'\
                                   ,'cemeteryName','plot_haystack','cemeteryId','missing_status','birthYear_needle','entry_state','soldiers_cemetery'\
                                   ,'plot_needle','birthCity','lastName','firstName','middleName','intermentHasPhoto'\
                                   ,'birthYear_haystack','birthStateName','birthCountryName','Siblings','soldiers_last_name','soldiers_first_name','soldiers_middle_name']
    
    parents_soldiers_child_match_df=parents_soldiers_child_match_df[parents_soldiers_match_df_list]
    parents_soldiers_child_match_df = parents_soldiers_child_match_df.sort_values(by=['name_index','fullName'],ascending=True)
    print("$$$$$  Soldier-Parent-1-2-Child Processing Complete $$$")
    return parents_soldiers_child_match_df

blank_line='                                                                                                                                                                                                  '
#&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&
child_process_flag=1
#&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&

maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
collection_date_dt=pd.to_datetime(collection_date0)
collection_date=collection_date0

start_time=datetime.datetime.now()
event_time=start_time

process_name='Match WWI Soldiers to Parents By Partitions'
print('**Starting *** ',process_name,' *** , Time:',start_time,' **\n')

DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"

fileABMCInput="ABMC_FAG_Soldiers_Merge_Master_WWI"
fileSoDInput="star_david_20251019_df"
fileRankInput="Rank_Dictionary"
fileInputCemetery="FAG_Cemeteries_Affiliation_20251205" 

fileABMCIn=DirMain+"0-Master Worksheets\\"+fileABMCInput+".csv"
fileSoDIn=DirMain+"0-Master Worksheets\\"+fileSoDInput+".csv"
fileRankIn=DirMain+"0-Master Worksheets\\"+fileRankInput+".csv"
fileInCemetery=DirMain+"0-Master Worksheets\\Cemeteries\\"+fileInputCemetery+".csv"

tic_start_total=get_tic()

soldiers_df=pd.read_csv(fileABMCIn,encoding='ISO-8859-1', on_bad_lines='skip',low_memory=False)
print('• soldiers_df * Download Complete * Size:',len(soldiers_df))
star_david_df=pd.read_csv(fileSoDIn)
rank_df=pd.read_csv(fileRankIn)
cemetery_df=pd.read_csv(fileInCemetery,encoding='ISO-8859-1', on_bad_lines='skip',low_memory=False)
print('• cemetery_df * Download Complete * Size:',len(cemetery_df))
      
rank_lookup = rank_df.set_index("abbreviated_rank")["rank"].to_dict()
soldiers_df = soldiers_df.dropna(subset=['Parents']) 

soldiers_df["soldiers_full_name"]=soldiers_df["soldiers_full_name"].str.lower()

soldiers_df["soldiers_full_name"]=soldiers_df["soldiers_full_name"].astype(str).replace(r' +',' ', regex=True)
soldiers_df["soldiers_full_name"]=soldiers_df["soldiers_full_name"].apply(strip_rank)


soldiers_df=drop_sets_dataframe_duplicates(soldiers_df)
soldiers_df.reset_index(drop=True, inplace = True)

soldiers_df=soldiers_df.copy()

soldiers_df['parent_1']=soldiers_df['Parents'].str.split('*').str[0].str.strip()
soldiers_df['parent_2']=soldiers_df['Parents'].str.split('*').str[1].str.strip()
soldiers_df['parent_1']=soldiers_df['parent_1'].apply(normalize)
soldiers_df['parent_2']=soldiers_df['parent_2'].apply(normalize)

# $$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$

group_soldier_parents_WW=['WWI','FAG_Relationship_Names_WWI_Graves_','Soldier_Parents_WWI',1,126]
group_soldier_parents_WWII=['WWI','FAG_Relationship_Names_WWI_Graves_','Soldier_Parents_WWI',1,126]
partition_df=pd.DataFrame([group_soldier_parents])
soldier_parent_match_partial_df=pd.DataFrame([])
soldier_parent_match_df=pd.DataFrame([])

partition_df_list=['directory','format','label','range_start','range_end']
partition_df.columns=[partition_df_list]

for index, row in partition_df.iterrows():
    tic_start=get_tic()
    tic=get_tic()
    subdirectory=row['directory']
    file_format=row['format']
    label=row['label']
    partition_start=row['range_start']
    partition_end=row['range_end']
    fileOutRelatives=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Outputs\\"+subdirectory+label+"_"+timestamp+".csv"

    start=partition_start
    end=partition_end+1
    parents_soldiers_match_df=pd.DataFrame([])

    # parents_df=pd.DataFrame([])
    tic=get_tic()
    for file_number in range(start,end):
        file_number_str='00'+str(file_number)
        file_number_str=file_number_str[-3:]
        # fileInRelatives=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Inputs\\"+subdirectory+"\\"+file_format+file_number_str+".csv" 
        fileInRelatives=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Inputs\\"+subdirectory+"\\"+file_format+file_number_str+".csv" 
        parents_partition_df=pd.read_csv(fileInRelatives,encoding='ISO-8859-1', on_bad_lines='skip',low_memory=False)
        print(f" {blank_line}", end='\r')
        print('• parents_partition_df * Download Complete * Size: ',len(parents_partition_df), end='\r')    
        
        # %%%%%%%%%%%%%%%%  Drop If No Parents's Entry %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
        parents_partition_df = parents_partition_df.dropna(subset=['Parents']) 

        parents_partition_df = parents_partition_df.drop_duplicates()
        parents_partition_df.reset_index(drop=True, inplace = True)
        parents_partition_df_file_size=len(parents_partition_df)
        # print('• parents_partition_df * Drop If No Children Entry * Size:',parents_partition_df_file_size) 
     
        parents_partial_df=parents_partition_df
        # print('• Normalize parents_partial_df *')
        parents_partial_df['Children']=parents_partial_df['Children'].str.lower()
        parents_partial_df['fullName']=parents_partial_df['fullName'].str.lower()     
        parents_partial_df['Spouses']=parents_partial_df['Spouses'].str.lower()   
        parents_partial_df['Children']= parents_partial_df['Children'].astype(str).replace(r'\.', '', regex=True).str.strip()
        parents_partial_df['Children']= parents_partial_df['Children'].astype(str).replace(r' +', ' ', regex=True).str.strip()
        parents_partial_df['Spouses']= parents_partial_df['Spouses'].astype(str).replace(r'\.', '', regex=True).str.strip()
        parents_partial_df['Spouses']= parents_partial_df['Spouses'].astype(str).replace(r' +', ' ', regex=True).str.strip()   
        
        # print("• Extract Children's Rank *")
        rank_temp=parents_partial_df["Children"].apply(extract_rank_2) 
        parents_partial_df[['children_rank', 'children_rank_2']] = pd.DataFrame(rank_temp.tolist())
        
        if 'plot' not in parents_partial_df.columns:parents_partial_df['plot']='No Plot Info'

        
        parents_short_df_list=['lastName','firstName','middleName','fullName','cemeteryName','plot','cemeteryId'\
                               ,'intermentHasPhoto','birthYear','birthStateName','birthCountryName','Spouses','Children','children_rank'\
                               ,'children_rank_2','Siblings']
        
        soldiers_short_df_list=['name_index','soldiers_last_name','soldiers_first_name','soldiers_middle_name','soldiers_full_name'\
                                ,'military_rank','missing_status' ,'intermentHasPhoto','birthYear','entry_state'\
                                  ,'soldiers_cemetery','plot','birthCity','birthState','birthCountry','parent_1','parent_2']
       
        parents_short_df=parents_partial_df[parents_short_df_list]
        soldiers_short_df=soldiers_df[soldiers_short_df_list]

       # ****************** Soldier-Parent 1 Processing   ****************************
        
        needle_df=soldiers_short_df
        haystack_df=parents_short_df

        # print(' *** soldier_parent1_match_partial_df Start ***')
        soldier_parent1_match_partial_df = needle_df.merge(
            haystack_df,
            left_on=['parent_1'],
            right_on=['fullName'],
            how="inner"
        )
        # print(' *** soldier_parent1_match_partial_df Complete ***')
        soldier_parent2_match_partial_df = needle_df.merge(
            haystack_df,
            left_on=['parent_2'],
            right_on=['fullName'],
            how="inner"
        )
        # print(' *** soldier_parent1_match_partial_df Complete ***')
        soldier_parent_match_partial_df=pd.concat([soldier_parent1_match_partial_df,soldier_parent2_match_partial_df])
        soldier_parent_match_partial_df = soldier_parent_match_partial_df.drop_duplicates()
        soldier_parent_match_partial_df.reset_index(drop=True, inplace = True)
        soldier_parent_match_partial_df_file_size=len(soldier_parent_match_partial_df)
        
        # print(' *** soldier_parent_match_partial_df Merge Complete ***')
        soldier_parent_match_partial_df['cemeteryId'] = pd.to_numeric(soldier_parent_match_partial_df['cemeteryId'], errors='coerce').astype('Int64')       
        cemetery_df_correlation=pd.merge(
        soldier_parent_match_partial_df,
        cemetery_df[['cemetery_id','prob_jewish','low_grave_count','memorial_count']],
        left_on=['cemeteryId'],
        right_on=['cemetery_id'],
        how='left')        
        
        soldier_parent_match_partial_df=cemetery_df_correlation.copy()        
        soldier_parent_match_partial_df.drop(columns=['cemetery_id'], inplace=True)        
        # print(' *** Cemetery Complete ***')
        soldier_parent_match_df=pd.concat([soldier_parent_match_df,soldier_parent_match_partial_df])
        soldier_parent_match_df = soldier_parent_match_df.drop_duplicates()
        soldier_parent_match_df.reset_index(drop=True, inplace = True)
        
        soldier_parent_match_df_file_size=len(soldier_parent_match_df)        


        
        file_parents_df=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Outputs\\"+subdirectory+"\\"+label+"_soldier_parent_match_df_"+timestamp+".csv"
        soldier_parent_match_df.to_csv(file_parents_df,mode="w",index=False,header=True)
        process_time= time_lapse(tic)
        process_time_cum= time_lapse(tic_start)
        tic=get_tic()          
        
        print(f" {blank_line}", end='\r')
        # print(f"$$$$  Soldier-Parent Processing Complete {len(jewish_parents_soldiers_match_df):,} - Process Time: {process_time/60:4.1f}/{process_time_cum/60:4.1f} Minutes", end='\r') 
        print(f"** Group: {label} • Uploading File {file_format+file_number_str} •  Partitions: {end} • Record Count: {parents_partition_df_file_size:,}/ {soldier_parent_match_partial_df_file_size:,}/ {soldier_parent_match_df_file_size:,} - Process Time: {process_time/60:4.1f}/{process_time_cum/60:4.1f} Minutes")    



    soldier_parent_match_final_df=soldier_parent_match_df.copy()
    soldier_parent_match_final_df_list_reorder=['name_index','soldiers_last_name','soldiers_first_name','soldiers_middle_name','soldiers_full_name','military_rank','parent_1','parent_2'\
                                          ,'fullName','Spouses','Children','children_rank','prob_jewish','cemeteryName','plot_y','cemeteryId'\
                                          ,'missing_status','entry_state','birthState','birthStateName','birthCountryName','soldiers_cemetery','birthYear_x','birthYear_y','intermentHasPhoto_x','intermentHasPhoto_y']
    
    soldier_parent_match_final_df_list_rename=['name_index','soldiers_last_name','soldiers_first_name','soldiers_middle_name','soldiers_full_name','military_rank','parent_1','parent_2'\
                                         ,'parent_fullName','Spouses','Children','children_rank','prob_jewish','parent_cemetery','parent_plot','parent_cemetery_id'\
                                         ,'missing_status','entry_state','soldiers_birth_state','parent_birth_state','parent_birth_country'\
                                         ,'soldiers_cemetery','soldiers_birth_year','parent_birth_year','soldiers_intermentHasPhoto','parent_intermentHasPhoto']
    
    soldier_parent_match_final_df=soldier_parent_match_final_df[soldier_parent_match_final_df_list_reorder]
    soldier_parent_match_final_df.columns=soldier_parent_match_final_df_list_rename   
    
    dataframe_df=soldier_parent_match_final_df
    needle='soldiers_full_name'
    haystack='Children'
    dataframe_df = dataframe_df.dropna(subset=['Children']) 
    dataframe_df.reset_index(drop=True, inplace = True)  
    soldier_parent_match_final_df,mask_children=haystack_contains_whole_needle_fuzzy(dataframe_df,needle,haystack)
    
    soldier_parent_match_final_df['Spouses_1']=soldier_parent_match_final_df['Spouses']
    dataframe_df=soldier_parent_match_final_df
    needle='parent_1'
    haystack='Spouses_1'
    dataframe_df = dataframe_df.dropna(subset=['parent_1']) 
    dataframe_df = dataframe_df.dropna(subset=['Spouses_1']) 
    dataframe_df.reset_index(drop=True, inplace = True)  
    
    soldier_parent_match_final_df,mask_parent_1=haystack_contains_whole_needle_fuzzy(dataframe_df,needle,haystack)
    
    soldier_parent_match_final_df['Spouses_2']=soldier_parent_match_final_df['Spouses']
    dataframe_df=soldier_parent_match_final_df
    needle='parent_2'
    haystack='Spouses_2'
    dataframe_df = dataframe_df.dropna(subset=['parent_2']) 
    dataframe_df = dataframe_df.dropna(subset=['Spouses_2']) 
    dataframe_df.reset_index(drop=True, inplace = True)  
    
    soldier_parent_match_final_df,mask_parent_2=haystack_contains_whole_needle_fuzzy(dataframe_df,needle,haystack)
    
    soldier_parent_match_final_df_list_rename=['name_index','Spouses_1_match','Spouses_2_match','Children_match','soldiers_last_name'\
                                         ,'soldiers_first_name','soldiers_middle_name','soldiers_full_name','military_rank','parent_1','parent_2'\
                                         ,'parent_fullName','Spouses','Children','children_rank','prob_jewish','parent_cemetery','parent_plot','parent_cemetery_id'\
                                         ,'missing_status','entry_state','soldiers_birth_state','parent_birth_state','parent_birth_country'\
                                         ,'soldiers_cemetery','soldiers_birth_year','parent_birth_year','soldiers_intermentHasPhoto','parent_intermentHasPhoto']
    
    soldier_parent_match_final_df=soldier_parent_match_final_df[soldier_parent_match_final_df_list_rename]
    
    soldier_parent_match_final_df['Children']=soldier_parent_match_final_df['Children'].str.title()
    soldier_parent_match_final_df['soldiers_full_name']=soldier_parent_match_final_df['soldiers_full_name'].str.title()     
    soldier_parent_match_final_df['Spouses']=soldier_parent_match_final_df['Spouses'].str.title()   
    soldier_parent_match_final_df['parent_1']=soldier_parent_match_final_df['parent_1'].str.title()
    soldier_parent_match_final_df['parent_2']=soldier_parent_match_final_df['parent_2'].str.title()     
    soldier_parent_match_final_df['parent_fullName']=soldier_parent_match_final_df['parent_fullName'].str.title()   
    
    file_parents_df=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Outputs\\"+subdirectory+"\\"+label+"_soldier_parent_match_final_df_"+timestamp+".csv"
    soldier_parent_match_final_df.to_csv(file_parents_df,mode="w",index=False,header=True)
    process_time= time_lapse(tic)
    process_time_cum= time_lapse(tic_start)
    tic=get_tic()          
    
    print(f" {blank_line}", end='\r')
    # print(f"$$$$  Soldier-Parent Processing Complete {len(jewish_parents_soldiers_match_df):,} - Process Time: {process_time/60:4.1f}/{process_time_cum/60:4.1f} Minutes", end='\r') 
    print(f"** Group: {label} • Uploading File {file_format+file_number_str} •  Partitions: {end} • Record Count: {parents_partition_df_file_size:,}/ {soldier_parent_match_partial_df_file_size:,}/ {soldier_parent_match_final_df_file_size:,} - Process Time: {process_time/60:4.1f}/{process_time_cum/60:4.1f} Minutes")    
        






 
process_time= time_lapse(tic)
process_time_cum= time_lapse(tic_start)
print(f"$$$$$$   Process {process_name} Complete  - Process Time: {process_time/60:4.1f}/{process_time_cum/60:4.1f} Minutes")

**Starting ***  Match WWI Soldiers to Parents By Partitions  *** , Time: 2025-12-10 16:21:53.945281  **

• soldiers_df * Download Complete * Size: 29271
• cemetery_df * Download Complete * Size: 251399
** Group: Soldier_Parents_WWI • Uploading File FAG_Relationship_Names_WWI_Graves_001 •  Partitions: 127 • Record Count: 62,566/ 405/ 405 - Process Time:  0.0/ 0.6 Minutes                         
** Group: Soldier_Parents_WWI • Uploading File FAG_Relationship_Names_WWI_Graves_002 •  Partitions: 127 • Record Count: 58,035/ 130/ 535 - Process Time:  0.0/ 1.1 Minutes                         
** Group: Soldier_Parents_WWI • Uploading File FAG_Relationship_Names_WWI_Graves_003 •  Partitions: 127 • Record Count: 70,344/ 263/ 798 - Process Time:  0.0/ 1.7 Minutes                         
** Group: Soldier_Parents_WWI • Uploading File FAG_Relationship_Names_WWI_Graves_004 •  Partitions: 127 • Record Count: 62,760/ 1,080/ 1,878 - Process Time:  0.0/ 2.3 Minutes                     
** Group: Sold

NameError: name 'soldier_parent_match_final_df_file_size' is not defined

In [27]:


soldier_parent_match_df_list_reorder=['name_index','soldiers_last_name','soldiers_first_name','soldiers_middle_name','soldiers_full_name','military_rank','parent_1','parent_2'\
                                      ,'fullName','Spouses','Children','children_rank','prob_jewish','cemeteryName','plot_y','cemeteryId'\
                                      ,'missing_status','entry_state','birthState','birthStateName','birthCountryName','soldiers_cemetery','birthYear_x','birthYear_y','intermentHasPhoto_x','intermentHasPhoto_y']

soldier_parent_match_df_list_rename=['name_index','soldiers_last_name','soldiers_first_name','soldiers_middle_name','soldiers_full_name','military_rank','parent_1','parent_2'\
                                     ,'parent_fullName','Spouses','Children','children_rank','prob_jewish','parent_cemetery','parent_plot','parent_cemetery_id'\
                                     ,'missing_status','entry_state','soldiers_birth_state','parent_birth_state','parent_birth_country'\
                                     ,'soldiers_cemetery','soldiers_birth_year','parent_birth_year','soldiers_intermentHasPhoto','parent_intermentHasPhoto']

soldier_parent_match_df=soldier_parent_match_df[soldier_parent_match_df_list_reorder]
soldier_parent_match_df.columns=soldier_parent_match_df_list_rename   

dataframe_df=soldier_parent_match_df
needle='soldiers_full_name'
haystack='Children'
dataframe_df = dataframe_df.dropna(subset=['Children']) 
dataframe_df.reset_index(drop=True, inplace = True)  
soldier_parent_match_df,mask_children=haystack_contains_whole_needle_fuzzy(dataframe_df,needle,haystack)


soldier_parent_match_df['Spouses']= soldier_parent_match_df['Spouses'].astype(str).replace(r'\.', '', regex=True).str.strip()
soldier_parent_match_df['Spouses']= soldier_parent_match_df['Spouses'].astype(str).replace(r' +', ' ', regex=True).str.strip()   

soldier_parent_match_df['Spouses_1']=soldier_parent_match_df['Spouses']
dataframe_df=soldier_parent_match_df
needle='parent_1'
haystack='Spouses_1'
dataframe_df = dataframe_df.dropna(subset=['parent_1']) 
dataframe_df = dataframe_df.dropna(subset=['Spouses_1']) 
dataframe_df.reset_index(drop=True, inplace = True)  

soldier_parent_match_df,mask_parent_1=haystack_contains_whole_needle_fuzzy(dataframe_df,needle,haystack)

soldier_parent_match_df['Spouses_2']=soldier_parent_match_df['Spouses']
dataframe_df=soldier_parent_match_df
needle='parent_2'
haystack='Spouses_2'
dataframe_df = dataframe_df.dropna(subset=['parent_2']) 
dataframe_df = dataframe_df.dropna(subset=['Spouses_2']) 
dataframe_df.reset_index(drop=True, inplace = True)  

soldier_parent_match_df,mask_parent_2=haystack_contains_whole_needle_fuzzy(dataframe_df,needle,haystack)

soldier_parent_match_df_list_rename=['name_index','Spouses_1_match','Spouses_2_match','Children_match','soldiers_last_name'\
                                     ,'soldiers_first_name','soldiers_middle_name','soldiers_full_name','military_rank','parent_1','parent_2'\
                                     ,'parent_fullName','Spouses','Children','children_rank','prob_jewish','parent_cemetery','parent_plot','parent_cemetery_id'\
                                     ,'missing_status','entry_state','soldiers_birth_state','parent_birth_state','parent_birth_country'\
                                     ,'soldiers_cemetery','soldiers_birth_year','parent_birth_year','soldiers_intermentHasPhoto','parent_intermentHasPhoto']

soldier_parent_match_df=soldier_parent_match_df[soldier_parent_match_df_list_rename]

soldier_parent_match_df['Children']=soldier_parent_match_df['Children'].str.title()
soldier_parent_match_df['soldiers_full_name']=soldier_parent_match_df['soldiers_full_name'].str.title()     
soldier_parent_match_df['Spouses']=soldier_parent_match_df['Spouses'].str.title()   
soldier_parent_match_df['parent_1']=soldier_parent_match_df['parent_1'].str.title()
soldier_parent_match_df['parent_2']=soldier_parent_match_df['parent_2'].str.title()     
soldier_parent_match_df['parent_fullName']=soldier_parent_match_df['parent_fullName'].str.title()   



file_parents_df=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Outputs\\"+subdirectory+"\\"+label+"_soldier_parent_match_df_new_"+timestamp+".csv"
soldier_parent_match_df.to_csv(file_parents_df,mode="w",index=False,header=True)
process_time= time_lapse(tic)
process_time_cum= time_lapse(tic_start)
tic=get_tic()          

print(f" {blank_line}", end='\r')
# print(f"$$$$  Soldier-Parent Processing Complete {len(jewish_parents_soldiers_match_df):,} - Process Time: {process_time/60:4.1f}/{process_time_cum/60:4.1f} Minutes", end='\r') 
print(f"** Group: {label} • Uploading File {file_format+file_number_str} •  Partitions: {end} • Record Count: {parents_partition_df_file_size:,}/ {soldier_parent_match_partial_df_file_size:,}/ {soldier_parent_match_df_file_size:,} - Process Time: {process_time/60:4.1f}/{process_time_cum/60:4.1f} Minutes")    


** Group: Soldier_Parents_WWI • Uploading File FAG_Relationship_Names_WWI_Graves_126 •  Partitions: 127 • Record Count: 2,074/ 3/ 31,560 - Process Time:  6.2/311.8 Minutes                        


In [24]:
# soldier_parent_match_df.drop(columns=['Spouses_match','Spouses_2_match'], inplace=True)  
list(soldier_parent_match_df)

['name_index',
 'Children_match',
 'soldiers_last_name',
 'soldiers_first_name',
 'soldiers_middle_name',
 'soldiers_full_name',
 'military_rank',
 'parent_1',
 'parent_2',
 'parent_fullName',
 'Spouses',
 'Children',
 'children_rank',
 'prob_jewish',
 'parent_cemetery',
 'parent_plot',
 'parent_cemetery_id',
 'missing_status',
 'entry_state',
 'soldiers_birth_state',
 'parent_birth_state',
 'parent_birth_country',
 'soldiers_cemetery',
 'soldiers_birth_year',
 'parent_birth_year',
 'soldiers_intermentHasPhoto',
 'parent_intermentHasPhoto']